# VisClick — Part D (step 5): train source-domain YOLOv8s

**Prerequisite this session:** `04_assemble_source.ipynb` has been run (the local dir `/content/source_train/` with `images/`, `labels/`, `data.yaml` must exist). `04` is fast on a 2nd session because its listings are cached under `/content/_unified_*.list`.

**This notebook** (`VisClick_Detailed_Plan.md` §D.1 + §D.2 + §D.3):
1. Mount Drive → `git pull` → GPU check → install Ultralytics.
2. Sanity-check that `/content/source_train/data.yaml` exists.
3. **Train YOLOv8s** on `source_train/data.yaml`. Writes to `<DRIVE>/weights/baseline_source/run1/` so it **survives runtime restarts**.
4. **Resume-aware:** if `run1/weights/best.pt` exists we skip training; if only `last.pt` exists we **resume**; else **fresh**. Override with `FORCE_FRESH = True`.
5. Save a stable name: `baseline_source/best_source_v8s.pt` (for later notebooks).
6. Sanity-predict on the source test split → `<DRIVE>/weights/baseline_source/predict1/`.
7. Write metrics CSV at `<DRIVE>/reports/tables/source_domain_results.csv` + copy training curves figure.

**Report:** every step prints `REPORT ...` lines for `VisClick_Report_Data_Form.md` §4 row *Source baseline*.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "opencv-python"], check=False)
import torch, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ultralytics:", ultralytics.__version__)
print("REPORT env | torch =", torch.__version__, "| cuda =", torch.cuda.is_available(), "| ultralytics =", ultralytics.__version__)

## 5.1 — Sanity: source_train is ready

If this fails, re-run `04_assemble_source.ipynb` first (a new Colab runtime wipes `/content/`).


In [ ]:
import os
SRC = "/content/source_train"
DATA_YAML = os.path.join(SRC, "data.yaml")

def _count(p):
    try:
        n = 0
        with os.scandir(p) as it:
            for e in it:
                if e.is_file(follow_symlinks=False) or e.is_symlink():
                    n += 1
        return n
    except OSError:
        return 0

assert os.path.isfile(DATA_YAML), f"Missing {DATA_YAML}. Run 04_assemble_source.ipynb first."
for sp in ("train", "val", "test"):
    img_n = _count(os.path.join(SRC, "images", sp))
    lbl_n = _count(os.path.join(SRC, "labels", sp))
    print(f"REPORT precheck | split = {sp:5s} | images = {img_n:>5d} | labels = {lbl_n:>5d}")
    assert img_n > 0 and lbl_n > 0, f"{sp}: empty images/labels — re-run 04"
print("data.yaml ->")
print(open(DATA_YAML).read())

## 5.2 — Train (fresh / resume / skip)

Logic:
- `best.pt` exists + not `FORCE_FRESH` → **skip** training, just load the best weights.
- `last.pt` exists → **resume**.
- else → fresh run.

Persistence: `project = <DRIVE>/weights/baseline_source` → Drive. Colab Free may disconnect after ~12 h; just re-run this cell, it will resume.


In [ ]:
import os, shutil, time
from ultralytics import YOLO

DRIVE = "/content/drive/MyDrive/visclick"
PROJECT = os.path.join(DRIVE, "weights", "baseline_source")
NAME    = "run1"
RUN_DIR = os.path.join(PROJECT, NAME)
LAST_PT = os.path.join(RUN_DIR, "weights", "last.pt")
BEST_PT = os.path.join(RUN_DIR, "weights", "best.pt")
os.makedirs(PROJECT, exist_ok=True)

FORCE_FRESH = False
EPOCHS      = 30
IMGSZ       = 640
BATCH       = 16

t0 = time.time()
train_status = None
results = None

if os.path.isfile(BEST_PT) and not FORCE_FRESH:
    print("best.pt already exists -> skipping training and loading:", BEST_PT)
    model = YOLO(BEST_PT)
    train_status = "SKIP_ALREADY_TRAINED"
elif os.path.isfile(LAST_PT) and not FORCE_FRESH:
    print("last.pt found -> resuming:", LAST_PT)
    model = YOLO(LAST_PT)
    results = model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=2,
        optimizer="AdamW", lr0=0.001, cos_lr=True, patience=10,
        project=PROJECT, name=NAME, exist_ok=True,
        cache=True, seed=0, save=True, plots=True, resume=True,
    )
    train_status = "RESUMED"
else:
    print("no prior checkpoint -> training fresh from yolov8s.pt")
    model = YOLO("yolov8s.pt")
    results = model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=2,
        optimizer="AdamW", lr0=0.001, cos_lr=True, patience=10,
        project=PROJECT, name=NAME, exist_ok=True,
        cache=True, seed=0, save=True, plots=True,
    )
    train_status = "FRESH"

elapsed = time.time() - t0
print(f"REPORT step = TRAIN | status = {train_status} | elapsed_s = {elapsed:0.0f} | run_dir = {RUN_DIR}")

## 5.3 — Save a stable name for `best.pt`

Later notebooks reference `baseline_source/best_source_v8s.pt` without needing to know the run number.


In [ ]:
import os, shutil
STABLE = os.path.join(PROJECT, "best_source_v8s.pt")
if os.path.isfile(BEST_PT):
    shutil.copy2(BEST_PT, STABLE)
    print("REPORT step = SAVE_STABLE | src =", BEST_PT, "| dst =", STABLE, "| bytes =", os.path.getsize(STABLE))
else:
    print("NOTE: best.pt missing; training may not have produced a best checkpoint yet.")
    print("REPORT step = SAVE_STABLE | status = MISSING_BEST")

## 5.4 — Sanity-check predictions on source test split

Runs `model.predict` on a small number of test images and saves annotated PNGs under `<DRIVE>/weights/baseline_source/predict1/`.


In [ ]:
import os, random, glob
random.seed(0)

pred_dir = os.path.join(PROJECT, "predict1")
test_imgs = sorted(glob.glob(os.path.join(SRC, "images", "test", "*.jpg"))) + \
            sorted(glob.glob(os.path.join(SRC, "images", "test", "*.png")))
if not test_imgs:
    print("REPORT step = PREDICT | status = NO_TEST_IMGS")
else:
    sample = random.sample(test_imgs, k=min(9, len(test_imgs)))
    res = model.predict(source=sample, conf=0.25, save=True,
                         project=PROJECT, name="predict1", exist_ok=True)
    print("REPORT step = PREDICT | n_images =", len(sample), "| out_dir =", pred_dir)

## 5.5 — Write metrics CSV for the report

Computes mAP@.5, mAP@.5:.95, precision, recall on the source test split and writes:
- `<DRIVE>/reports/tables/source_domain_results.csv` (one-row summary for §4 of the data form).
- Copies training curves PNG to `<DRIVE>/reports/figures/source_train_curves.png` if Ultralytics saved one.


In [ ]:
import os, csv, shutil, glob, time
TABLES = os.path.join(DRIVE, "reports", "tables")
FIGS   = os.path.join(DRIVE, "reports", "figures")
os.makedirs(TABLES, exist_ok=True); os.makedirs(FIGS, exist_ok=True)

val_m = model.val(data=DATA_YAML, split="test", project=PROJECT, name="val_test", exist_ok=True, plots=False, verbose=False)
box = getattr(val_m, "box", None)
row = {
    "model": "YOLOv8s",
    "dataset": "source_train (unified-sampled, 6-class)",
    "n_train": _count(os.path.join(SRC, "images", "train")),
    "n_val":   _count(os.path.join(SRC, "images", "val")),
    "n_test":  _count(os.path.join(SRC, "images", "test")),
    "imgsz": IMGSZ, "batch": BATCH, "epochs": EPOCHS,
    "map50":     float(getattr(box, "map50", float('nan'))) if box is not None else float('nan'),
    "map50_95":  float(getattr(box, "map",   float('nan'))) if box is not None else float('nan'),
    "precision": float(getattr(box, "mp",    float('nan'))) if box is not None else float('nan'),
    "recall":    float(getattr(box, "mr",    float('nan'))) if box is not None else float('nan'),
    "weights": STABLE if os.path.isfile(STABLE) else BEST_PT,
    "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
}
csv_path = os.path.join(TABLES, "source_domain_results.csv")
with open(csv_path, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)
print("REPORT step = METRICS | csv =", csv_path)
for k, v in row.items():
    print(f"REPORT metric | {k:10s} = {v}")

curve_src = os.path.join(RUN_DIR, "results.png")
if os.path.isfile(curve_src):
    dst = os.path.join(FIGS, "source_train_curves.png")
    shutil.copy2(curve_src, dst)
    print("REPORT step = FIG_COPY | src =", curve_src, "| dst =", dst)

**Next:** `notebooks/06_finetune_desktop.ipynb` (§E) — head-only fine-tune on your ~50 desktop screenshots, starting from `best_source_v8s.pt`.
